In [1]:
import pandas as pd

# Load your dataframe
df = pd.read_csv('../data/final.csv')


In [3]:
print(df.columns.tolist())


['CountryCode', 'Amount', 'Value', 'PricingStrategy', 'FraudResult', 'TotalTransactionAmount', 'AverageTransactionAmount', 'TransactionCount', 'StdDevTransactionAmount', 'MaxTransactionAmount', 'TransactionHour', 'TransactionDay', 'TransactionMonth', 'TransactionYear', 'TransactionId_label', 'BatchId_label', 'AccountId_label', 'SubscriptionId_label', 'CustomerId_label', 'CurrencyCode_label', 'ProviderId_label', 'ProductId_label', 'ProductCategory_label', 'ChannelId_label']


In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load dataset
df = pd.read_csv('../data/final.csv')

# Step 1: Proxy "Transaction Time" using TransactionId_label as a sequence
df['TransactionSeq'] = pd.to_numeric(df['TransactionId_label'], errors='coerce')

# Snapshot = max TransactionSeq
snapshot_seq = df['TransactionSeq'].max()
print("🕒 Snapshot (max TransactionSeq):", snapshot_seq)

# Step 2: Calculate RFM with TransactionSeq as proxy for time
rfm = df.groupby('CustomerId_label').agg({
    'TransactionSeq': lambda x: snapshot_seq - x.max(),  # Recency
    'TransactionId_label': 'count',                      # Frequency
    'Amount': 'sum'                                      # Monetary
}).reset_index()

rfm.rename(columns={
    'TransactionSeq': 'Recency',
    'TransactionId_label': 'Frequency',
    'Amount': 'Monetary'
}, inplace=True)

# Clean RFM
rfm = rfm.dropna(subset=['Recency', 'Frequency', 'Monetary'])
print("✅ RFM shape:", rfm.shape)

# Step 3: Scale and cluster
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=3, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Step 4: Identify high-risk cluster (high recency, low freq/monetary)
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
print("\n📊 Cluster Summary:")
print(cluster_summary)

high_risk_cluster = cluster_summary.sort_values(
    ['Recency', 'Frequency', 'Monetary'],
    ascending=[False, True, True]
).index[0]

rfm['is_high_risk'] = (rfm['Cluster'] == high_risk_cluster).astype(int)

# Step 5: Merge back into main df
df = df.merge(rfm[['CustomerId_label', 'is_high_risk']], on='CustomerId_label', how='left')
df['is_high_risk'] = df['is_high_risk'].fillna(0).astype(int)

# Step 6: Save final file
df.to_csv('../data/with_risk.csv', index=False)
print("✅ Saved to: ../data/with_high_risk.csv")
print("✅ is_high_risk counts:\n", df['is_high_risk'].value_counts())


🕒 Snapshot (max TransactionSeq): 1.0
✅ RFM shape: (3742, 4)

📊 Cluster Summary:
          Recency    Frequency    Monetary
Cluster                                   
0        0.081490    29.167401    2.703090
1        0.000094  2681.666667  243.454729
2        0.604220     1.959391    0.183183
✅ Saved to: ../data/with_high_risk.csv
✅ is_high_risk counts:
 is_high_risk
0    94118
1     1544
Name: count, dtype: int64
